<a href="https://colab.research.google.com/github/REVREBEL/Metrics-Library/blob/main/notebooks/process_pace_data_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Imports & Definitions

In [12]:
import glob
import os, shutil, re
import pandas as pd
import numpy as np
import pandas_gbq
from google.cloud import bigquery
from google.colab import drive
import types
import requests

# --- 1. CONFIGURATION (Master Variables) ---
global PROJECT_ID, DATASET_ID, PACE_PROPERTY_TABLE, PACE_SEGMENT_TABLE, PACE_ROOMTYPE_TABLE, DIM_PROPERTY_TABLE, SOURCE_DIR, NEW_FILES_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR

PROJECT_ID = "devrebel-big-query-database"
DATASET_ID = "dev_hotel_analytics"
PACE_PROPERTY_TABLE = "snap_pace_property"
PACE_SEGMENT_TABLE = "snap_pace_segment"
PACE_ROOMTYPE_TABLE = "snap_pace_roomtype"
DIM_PROPERTY_TABLE = "dim_property"

SOURCE_DIR = "/content/drive/Shareddrives/Data/DEVREB/pace-data/"
NEW_FILES_DIR = os.path.join(SOURCE_DIR, "Data_Upload")
PROCESSED_DIR = os.path.join(SOURCE_DIR, "Data_Processed")
EXPORT_DIR = os.path.join(SOURCE_DIR, "Data_Export")
FAILED_DIR = os.path.join(SOURCE_DIR, "Data_Failed")

# --- 2. SHEET CONFIGURATIONS (Standard Funnel) ---
sheet_configs = {
    "Property": {"source_report": "snap_property", "target_table": "fact_pace_property", "source_system": "IDeaS", "extra_map": {}},
    "Room Type": {"source_report": "snap_pace_roomtype", "target_table": "fact_pace_roomtype", "source_system": "IDeaS", "extra_map": {}},
    "Room Class": {"source_report": "snap_pace_roomclass", "target_table": "fact_pace_roomclass", "source_system": "IDeaS", "extra_map": {"room_class": "roomclass"}},
    "Business View": {"source_report": "snap_pace_segment", "target_table": "fact_pace_segment", "source_system": "IDeaS", "extra_map": {"business_view": "segment"}},

}

# --- 3. INITIALIZATION HELPERS ---
def setup_environment():
    drive.mount('/content/drive', force_remount=True)
    for d in [PROCESSED_DIR, EXPORT_DIR, NEW_FILES_DIR, FAILED_DIR]:
        os.makedirs(d, exist_ok=True)
    print(f"✅ Environment initialized.")

def get_property_code_from_file_name(file_name, dim_property_df):
    code_match = re.search(r'([A-Z]{3}[A-Z0-9]{3})', file_name, re.IGNORECASE)
    if code_match:
        extracted_code = code_match.group(1).upper()
        if extracted_code in dim_property_df['property_code'].values: return extracted_code
    return 'UNKNOWN'

# --- 4. EXTERNAL HELPERS (Standardizer) ---
url = "https://raw.githubusercontent.com/REVREBEL/Metrics-Library/main/notebooks/utilities/revrebel_column_standardizer.py"
module_code = requests.get(url).text
revrebel_standardizer = types.ModuleType("revrebel_column_standardizer")
exec(module_code, revrebel_standardizer.__dict__)
standardize_dataframe = revrebel_standardizer.standardize_dataframe

# --- 5. MASTER PIPELINE ---
def run_master_pipeline():
    setup_environment()
    print("Loading dim_property from BigQuery...")
    dim_property_df = pandas_gbq.read_gbq(f"SELECT property_code FROM `{PROJECT_ID}.{DATASET_ID}.{DIM_PROPERTY_TABLE}`", project_id=PROJECT_ID)

    files_to_process = glob.glob(os.path.join(NEW_FILES_DIR, "*.xlsx"))
    print(f"Found {len(files_to_process)} files to process.")

    for file_path in files_to_process:
        file_name = os.path.basename(file_path)
        file_base = os.path.splitext(file_name)[0]
        print(f"\n🚀 Processing: {file_name}")

        date_match = re.search(r'(\d{8})', file_name)
        if not date_match: continue
        snap_date_str = pd.to_datetime(date_match.group(1), format='%Y%m%d').strftime('%Y-%m-%d')
        prop_code = get_property_code_from_file_name(file_name, dim_property_df)

        try:
            xls = pd.ExcelFile(file_path)
            for sheet_name, cfg in sheet_configs.items():
                if sheet_name in xls.sheet_names:
                    df = pd.read_excel(xls, sheet_name=sheet_name)
                    df_std = standardize_dataframe(
                        df,
                        source_report=cfg["source_report"],
                        extra_map=cfg["extra_map"],
                        metadata={
                            "property_code": prop_code,
                            "source_system": cfg.get("source_system", "IDeaS"),
                            "source_report": cfg["source_report"],
                            "source_file": file_name,
                            "snap_date": snap_date_str
                        },
                        print_report=False
                    )

                    # --- CLEANUP: Remove Dummy Rows ---
                    # Filter out the 'Dummy for excel' entry if present in any column
                    dummy_str = "Dummy for excel"
                    df_std = df_std[~df_std.astype(str).apply(lambda x: x.str.contains(dummy_str, case=False, na=False)).any(axis=1)]

                    csv_name = f"{file_base}_{sheet_name.replace(' ', '')}.csv"
                    df_std.to_csv(os.path.join(EXPORT_DIR, csv_name), index=False)
                    print(f"  ✅ {sheet_name} standardized and cleaned (dummies removed).")

            shutil.move(file_path, os.path.join(PROCESSED_DIR, file_name))
            print(f"✅ Finished: Original file moved to /Data_Processed")
        except Exception as e:
            print(f"❌ Error processing {file_name}: {e}")

if __name__ == "__main__":
    run_master_pipeline()

Mounted at /content/drive
✅ Environment initialized.
Loading dim_property from BigQuery...
Downloading: 100%|██████████|
Found 222 files to process.

🚀 Processing: DTWDFH_PaceData_20260404.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✅ Property standardized and cleaned (dummies removed).
  ✅ Room Type standardized and cleaned (dummies removed).
  ✅ Room Class standardized and cleaned (dummies removed).
  ✅ Business View standardized and cleaned (dummies removed).
✅ Finished: Original file moved to /Data_Processed

🚀 Processing: DTWDFH_PaceData_20260406.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✅ Property standardized and cleaned (dummies removed).
  ✅ Room Type standardized and cleaned (dummies removed).
  ✅ Room Class standardized and cleaned (dummies removed).
  ✅ Business View standardized and cleaned (dummies removed).
✅ Finished: Original file moved to /Data_Processed

🚀 Processing: DTWDFH_PaceData_20260408.xlsx


...

  ✅ Property standardized and cleaned (dummies removed).
  ✅ Room Type standardized and cleaned (dummies removed).
  ✅ Room Class standardized and cleaned (dummies removed).
  ✅ Business View standardized and cleaned (dummies removed).
✅ Finished: Original file moved to /Data_Processed

🚀 Processing: DTWDFH_PaceData_20250801.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✅ Property standardized and cleaned (dummies removed).
  ✅ Room Type standardized and cleaned (dummies removed).
  ✅ Room Class standardized and cleaned (dummies removed).
  ✅ Business View standardized and cleaned (dummies removed).
✅ Finished: Original file moved to /Data_Processed

🚀 Processing: DTWDFH_PaceData_20250807.xlsx


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  ✅ Property standardized and cleaned (dummies removed).
  ✅ Room Type standardized and cleaned (dummies removed).
  ✅ Room Class standardized and cleaned (dummies removed).
  ✅ Business View standardized and cleaned (dummies removed).
✅ Finished: Original file moved to /Data_Processed


# Setup & Auth

In [ ]:
# --- SETUP & AUTH ---
drive.mount('/content/drive')

# Create the processed folder if it doesn't exist
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)
    print(f"Created directory: {PROCESSED_DIR}")

print(f"Checking for new files in: {NEW_FILES_DIR}")

# Initialize the BigQuery client
client = initialize_client(PROJECT_ID)
print("BigQuery client initialized.")

# Mapping Refresh

In [ ]:
# --- SECTION 2: Mapping Refresh ---
# This section has been removed as per user request to handle mapping updates
# in a separate workflow. If you need to re-enable it, please refer to previous
# versions of the notebook.

# Get Property Code

In [ ]:
# Content moved to cell zDkwBwlw-CyM for single point of entry.

# Property Data Processing

In [ ]:
# Content moved to cell zDkwBwlw-CyM for single point of entry.

# Room Type Data Processing

In [ ]:
# Content moved to cell zDkwBwlw-CyM for single point of entry.

# Segment Data Processing


In [ ]:
# Content moved to cell zDkwBwlw-CyM for single point of entry.

# MASTER CONTROLLER

In [ ]:
# Master Controller has been consolidated into the 'Imports & Definitions' cell.
# Use this cell only for manual overrides or re-triggering the pipeline.
# run_master_pipeline()

# dim_property


In [ ]:
import bigframes.pandas as bf

bf.options.bigquery.location = "us-central1"
bf.options.bigquery.project = "devrebel-big-query-database"

df = bf.read_gbq("devrebel-big-query-database.dev_hotel_analytics.dim_property")
df.head(20)


In [ ]:
# Content moved to cell zDkwBwlw-CyM for single point of entry.

In [ ]:
import types
import requests

url = "https://raw.githubusercontent.com/REVREBEL/Metrics-Library/main/notebooks/utilities/revrebel_column_standardizer.py"

module_code = requests.get(url).text

revrebel_standardizer = types.ModuleType("revrebel_column_standardizer")
exec(module_code, revrebel_standardizer.__dict__)

standardize_dataframe = revrebel_standardizer.standardize_dataframe
get_column_report = revrebel_standardizer.get_column_report
print_column_report = revrebel_standardizer.print_column_report


# Read your CSV or Excel file into df

from google.colab import files
import pandas as pd

print("Please upload your CSV or Excel file.")
uploaded = files.upload()

if not uploaded:
    print("No file was uploaded. Please upload a file to proceed.")
else:
    file_name = list(uploaded.keys())[0]

    if file_name.endswith(".csv"):
        df = pd.read_csv(file_name)
    else:
        df = pd.read_excel(file_name)

    print_column_report(df, source_report="snap_pace_segment")


    # Standardize it

    df_standardized = standardize_dataframe(
        df,
        source_report="snap_pace_segment",
        metadata={
            "property_code": "DTWDFH", # You may need to change this property_code
            "source_system": "Duetto",
            "source_file": file_name,
        },
        print_report=True
    )

    print("\nNew column names after standardization:")
    print(df_standardized.columns.tolist())